# Texture Analysis Pipeline v3 - Balanced Training (Fixed)

**Düzeltmeler:**
- Custom WeightedModel kaldırıldı (metric sorunu)
- Oversampling ile dengeleme (daha güvenilir)
- Stratified batching
- Hedef MAE: 5-8

In [ ]:
!nvidia-smi
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
!pip install -q mediapipe PyWavelets scikit-image kaggle

import os, json
kaggle_credentials = {"username": "metekizilcik", "key": ""}
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle API hazir!")

In [ ]:
import os, shutil, random
from glob import glob

DATASET_SIZE = 10000
RAW_DIR = "raw_ffhq"

if not os.path.exists(RAW_DIR) or len(os.listdir(RAW_DIR)) < 100:
    print("FFHQ-256 indiriliyor...")
    !kaggle datasets download -d greatgamedota/ffhq-face-data-set -p /content/ffhq_download --unzip
    
    os.makedirs(RAW_DIR, exist_ok=True)
    all_images = glob("/content/ffhq_download/**/*.png", recursive=True)
    selected = random.sample(all_images, min(DATASET_SIZE, len(all_images)))
    
    for i, img_path in enumerate(selected):
        shutil.copy(img_path, os.path.join(RAW_DIR, os.path.basename(img_path)))
        if (i+1) % 2000 == 0: print(f"{i+1}/{len(selected)}")
    
    shutil.rmtree("/content/ffhq_download", ignore_errors=True)
print(f"{len(os.listdir(RAW_DIR))} gorsel hazir!")

In [ ]:
%%writefile feature_extractor_v3.py
import cv2
import numpy as np
import pywt
from scipy.ndimage import uniform_filter
from skimage.feature import graycomatrix, graycoprops

class TextureFeatureExtractor:
    def compute_entropy(self, image):
        if len(image.shape) == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        f_transform = np.fft.fft2(image.astype(np.float64))
        f_shift = np.fft.fftshift(f_transform)
        magnitude = np.abs(f_shift)
        magnitude_sum = magnitude.sum()
        if magnitude_sum == 0: return 0.0
        nfp = magnitude / magnitude_sum
        nfp_nonzero = nfp[nfp > 0]
        return -np.sum(nfp_nonzero * np.log2(nfp_nonzero))
    
    def compute_spatial_entropy(self, image):
        if len(image.shape) == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        hist, _ = np.histogram(image.flatten(), bins=256, range=(0, 256), density=True)
        hist_nonzero = hist[hist > 0]
        return -np.sum(hist_nonzero * np.log2(hist_nonzero))
    
    def compute_dwt_energy(self, image, wavelet='sym4', level=3):
        if len(image.shape) == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        coeffs = pywt.wavedec2(image.astype(np.float64), wavelet, level=level)
        energy = sum(np.sum(s**2) for detail in coeffs[1:] for s in detail)
        return energy / (image.shape[0] * image.shape[1])
    
    def compute_roughness(self, image, mask=None):
        if len(image.shape) == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        pixels = image[mask > 0] if mask is not None else image.flatten()
        if len(pixels) == 0: return 0.0, 0.0
        pixels = pixels.astype(np.float64)
        mean_val = np.mean(pixels)
        dev = pixels - mean_val
        return np.mean(np.abs(dev)), np.sqrt(np.mean(dev**2))
    
    def compute_mlc(self, image, window_size=15, mask=None):
        if len(image.shape) == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = image.astype(np.float64)
        local_mean = uniform_filter(image, size=window_size, mode='reflect')
        local_sq_mean = uniform_filter(image**2, size=window_size, mode='reflect')
        local_std = np.sqrt(np.maximum(local_sq_mean - local_mean**2, 0))
        valid = local_std[mask > 0] if mask is not None else local_std.flatten()
        return np.mean(valid) if len(valid) > 0 else 0.0
    
    def compute_glcm_features(self, image):
        if len(image.shape) == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = cv2.resize(image, (128, 128))
        image_q = (image // 4).astype(np.uint8)
        glcm = graycomatrix(image_q, distances=[1, 3], angles=[0, np.pi/4, np.pi/2],
                           levels=64, symmetric=True, normed=True)
        return graycoprops(glcm, 'contrast').mean(), graycoprops(glcm, 'homogeneity').mean()
    
    def extract_feature_vector(self, texture_map, mask=None):
        if len(texture_map.shape) == 3:
            texture_map = cv2.cvtColor(texture_map, cv2.COLOR_BGR2GRAY)
        Ra, Rq = self.compute_roughness(texture_map, mask)
        contrast, homogeneity = self.compute_glcm_features(texture_map)
        return np.array([
            self.compute_entropy(texture_map),
            self.compute_spatial_entropy(texture_map),
            self.compute_dwt_energy(texture_map),
            Ra, Rq,
            self.compute_mlc(texture_map, mask=mask),
            contrast, homogeneity
        ])
    
    @staticmethod
    def get_feature_names():
        return ['fourier_entropy', 'spatial_entropy', 'dwt_energy', 'Ra', 'Rq', 'mlc', 'glcm_contrast', 'glcm_homogeneity']

print("feature_extractor_v3.py OK")

In [ ]:
%%writefile label_generator_v3.py
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
import joblib

class LabelGenerator:
    def __init__(self):
        self.scaler = None
        self.pca = None
        self.score_min = None
        self.score_max = None
    
    def fit(self, feature_matrix):
        print(f"Fitting on {feature_matrix.shape[0]} samples")
        
        self.scaler = RobustScaler()
        scaled = self.scaler.fit_transform(feature_matrix)
        
        self.pca = PCA(n_components=1)
        degradation = self.pca.fit_transform(scaled).flatten()
        
        print(f"PCA Variance: {self.pca.explained_variance_ratio_[0]:.2%}")
        
        self.score_min = np.percentile(degradation, 2)
        self.score_max = np.percentile(degradation, 98)
        
        scores = self._to_score(degradation)
        print(f"Scores: min={scores.min():.1f}, max={scores.max():.1f}, mean={scores.mean():.1f}, std={scores.std():.1f}")
        return scores
    
    def _to_score(self, degradation):
        range_val = self.score_max - self.score_min
        if range_val == 0: return np.full_like(degradation, 50.0)
        normalized = (degradation - self.score_min) / range_val
        stretched = 1 / (1 + np.exp(-5 * (normalized - 0.5)))
        scores = (1 - stretched) * 100
        return np.clip(scores, 0, 100)
    
    def transform(self, feature_matrix):
        scaled = self.scaler.transform(feature_matrix)
        degradation = self.pca.transform(scaled).flatten()
        return self._to_score(degradation)
    
    def save(self, path):
        joblib.dump({'scaler': self.scaler, 'pca': self.pca,
                    'score_min': self.score_min, 'score_max': self.score_max}, path)
    
    def load(self, path):
        data = joblib.load(path)
        self.scaler, self.pca = data['scaler'], data['pca']
        self.score_min, self.score_max = data['score_min'], data['score_max']

print("label_generator_v3.py OK")

In [ ]:
%%writefile dataset_generator_v3.py
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os, urllib.request
from glob import glob
import pandas as pd
from feature_extractor_v3 import TextureFeatureExtractor
from label_generator_v3 import LabelGenerator

FACE_OVAL = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103]
LIPS = [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 185, 40, 39, 37, 0, 267, 269, 270, 409]
LEFT_EYE = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
RIGHT_EYE = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
LEFT_EYEBROW = [46, 53, 52, 65, 55, 70, 63, 105, 66, 107]
RIGHT_EYEBROW = [276, 283, 282, 295, 285, 300, 293, 334, 296, 336]

class DatasetGenerator:
    def __init__(self, input_dir="raw_ffhq", output_dir="training_data/images"):
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.detector = None
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs("training_data", exist_ok=True)
    
    def _init_detector(self):
        if self.detector: return
        model_path = 'face_landmarker.task'
        if not os.path.exists(model_path):
            urllib.request.urlretrieve(
                "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
                model_path)
        self.detector = vision.FaceLandmarker.create_from_options(
            vision.FaceLandmarkerOptions(
                base_options=python.BaseOptions(model_asset_path=model_path),
                num_faces=1))
    
    def _get_coords(self, landmarks, indices, w, h):
        return np.array([[int(landmarks[i].x * w), int(landmarks[i].y * h)] for i in indices])
    
    def process_image(self, img_path):
        self._init_detector()
        img = cv2.imread(img_path)
        if img is None: return None, None
        img = cv2.resize(img, (512, 512))
        h, w = 512, 512
        
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        result = self.detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        if not result.face_landmarks: return None, None
        
        landmarks = result.face_landmarks[0]
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [self._get_coords(landmarks, FACE_OVAL, w, h)], 255)
        for region in [LIPS, LEFT_EYE, RIGHT_EYE, LEFT_EYEBROW, RIGHT_EYEBROW]:
            cv2.fillPoly(mask, [self._get_coords(landmarks, region, w, h)], 0)
        
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edges = np.uint8(np.absolute(cv2.Laplacian(gray, cv2.CV_64F)))
        hair_zones = cv2.GaussianBlur(edges, (25, 25), 0)
        _, beard_mask = cv2.threshold(hair_zones, 25, 255, cv2.THRESH_BINARY)
        mask = cv2.bitwise_and(mask, cv2.bitwise_not(beard_mask))
        
        if np.sum(mask == 255) < 20000: return None, None
        
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l_channel = cv2.split(lab)[0]
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced_l = clahe.apply(l_channel)
        blurred = cv2.GaussianBlur(enhanced_l, (37, 37), 0)
        texture_map = cv2.add(cv2.subtract(enhanced_l, blurred), 128)
        
        return np.where(mask == 255, texture_map, 128).astype(np.uint8), mask
    
    def generate_dataset(self):
        print("="*60)
        print("DATASET GENERATION v3")
        print("="*60)
        
        image_paths = glob(os.path.join(self.input_dir, "*.png")) + glob(os.path.join(self.input_dir, "*.jpg"))
        print(f"Found {len(image_paths)} images")
        
        extractor = TextureFeatureExtractor()
        results, features_list = [], []
        
        for idx, img_path in enumerate(image_paths):
            texture_map, mask = self.process_image(img_path)
            if texture_map is None: continue
            
            filename = os.path.basename(img_path)
            cv2.imwrite(os.path.join(self.output_dir, filename), texture_map)
            features_list.append(extractor.extract_feature_vector(texture_map, mask))
            results.append(filename)
            
            if (idx + 1) % 1000 == 0:
                print(f"Progress: {idx+1}/{len(image_paths)} | Success: {len(results)}")
        
        print(f"Generated {len(results)} texture maps")
        if not results: return None
        
        feature_matrix = np.array(features_list)
        pd.DataFrame(feature_matrix, columns=extractor.get_feature_names()).assign(
            filename=results).to_csv('training_data/features.csv', index=False)
        
        generator = LabelGenerator()
        scores = generator.fit(feature_matrix)
        generator.save('training_data/pca_scaler.joblib')
        
        labels_df = pd.DataFrame({'filename': results, 'roughness_score': np.round(scores, 2)})
        labels_df.to_csv('training_data/labels.csv', index=False)
        
        return labels_df

print("dataset_generator_v3.py OK")

In [ ]:
%%writefile train_model_v3.py
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
import math

# GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def oversample_dataframe(df, target_per_bin=1500, n_bins=10):
    """Az ornekli skor araliklarini cogaltarak dengele"""
    bins = np.linspace(0, 100, n_bins + 1)
    df['bin'] = pd.cut(df['roughness_score'], bins=bins, labels=False, include_lowest=True)
    
    print("\nOriginal distribution:")
    bin_counts = df['bin'].value_counts().sort_index()
    for i, count in bin_counts.items():
        print(f"  {bins[i]:.0f}-{bins[i+1]:.0f}: {count}")
    
    balanced_dfs = []
    for bin_idx in range(n_bins):
        bin_df = df[df['bin'] == bin_idx]
        if len(bin_df) == 0:
            continue
        
        if len(bin_df) < target_per_bin:
            # Oversample
            n_repeats = int(np.ceil(target_per_bin / len(bin_df)))
            oversampled = pd.concat([bin_df] * n_repeats, ignore_index=True)
            balanced_dfs.append(oversampled.head(target_per_bin))
        else:
            # Undersample (veya aynen birak)
            balanced_dfs.append(bin_df.sample(n=min(len(bin_df), target_per_bin * 2), random_state=42))
    
    balanced_df = pd.concat(balanced_dfs, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle
    
    print(f"\nBalanced: {len(df)} -> {len(balanced_df)} samples")
    print("\nNew distribution:")
    new_counts = balanced_df['bin'].value_counts().sort_index()
    for i, count in new_counts.items():
        print(f"  {bins[i]:.0f}-{bins[i+1]:.0f}: {count}")
    
    return balanced_df.drop('bin', axis=1)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation")

def create_dataset(df, img_dir, batch_size=32, augment=False, shuffle=True):
    def process(filepath, score):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [224, 224])
        img = tf.cast(img, tf.float32)
        return img, score
    
    ds = tf.data.Dataset.from_tensor_slices((
        df['filepath'].values,
        df['roughness_score'].values.astype(np.float32)
    ))
    
    if shuffle:
        ds = ds.shuffle(len(df), reshuffle_each_iteration=True)
    
    ds = ds.map(process, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                   num_parallel_calls=tf.data.AUTOTUNE)
    
    return ds.prefetch(tf.data.AUTOTUNE)

def train(csv_path="training_data/labels.csv", img_dir="training_data/images",
          batch_size=64, epochs=40):
    print("="*60)
    print("MODEL TRAINING v3 - Balanced (Fixed)")
    print("="*60)
    
    df = pd.read_csv(csv_path)
    df['filepath'] = df['filename'].apply(lambda x: os.path.join(img_dir, x))
    df = df[df['filepath'].apply(os.path.exists)]
    
    print(f"\nOriginal: {len(df)} samples")
    print(f"Score stats: mean={df['roughness_score'].mean():.1f}, std={df['roughness_score'].std():.1f}")
    
    # Oversample ile dengele
    balanced_df = oversample_dataframe(df, target_per_bin=1200, n_bins=10)
    
    # Stratified split
    balanced_df['score_bin'] = pd.cut(balanced_df['roughness_score'], bins=10, labels=False)
    train_df, val_df = train_test_split(
        balanced_df, test_size=0.15, random_state=42, stratify=balanced_df['score_bin']
    )
    train_df = train_df.drop('score_bin', axis=1)
    val_df = val_df.drop('score_bin', axis=1)
    
    print(f"\nTraining: {len(train_df)}, Validation: {len(val_df)}")
    
    train_ds = create_dataset(train_df, img_dir, batch_size, augment=True)
    val_ds = create_dataset(val_df, img_dir, batch_size, augment=False, shuffle=False)
    
    # Model
    base = tf.keras.applications.EfficientNetB0(
        input_shape=(224, 224, 3), include_top=False, weights='imagenet'
    )
    
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=True)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(1, activation='linear', dtype='float32')(x)
    
    model = tf.keras.Model(inputs, outputs)
    
    # LR Schedule with warmup
    initial_lr = 1e-3
    def lr_schedule(epoch):
        if epoch < 3:
            return initial_lr * (epoch + 1) / 3
        return initial_lr * 0.5 * (1 + math.cos(math.pi * (epoch - 3) / (epochs - 3)))
    
    callbacks = [
        tf.keras.callbacks.LearningRateScheduler(lr_schedule),
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_mae'),
        tf.keras.callbacks.ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_mae'),
    ]
    
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=initial_lr, weight_decay=1e-5),
        loss=tf.keras.losses.Huber(delta=10.0),
        metrics=['mae']
    )
    
    print(f"\nTraining for {epochs} epochs...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks
    )
    
    # Load best and save
    model = tf.keras.models.load_model('best_model.keras')
    model.save('texture_regressor.keras')
    
    print(f"\nBest val_mae: {min(history.history['val_mae']):.2f}")
    print("Model saved: texture_regressor.keras")
    
    return model, history

print("train_model_v3.py OK")

In [ ]:
# Dataset Generation (SADECE FFHQ yoksa calistir)
import os
if not os.path.exists('training_data/labels.csv'):
    from dataset_generator_v3 import DatasetGenerator
    import matplotlib.pyplot as plt

    generator = DatasetGenerator(input_dir="raw_ffhq")
    labels_df = generator.generate_dataset()

    if labels_df is not None:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(labels_df['roughness_score'], bins=50, edgecolor='black')
        axes[0].set_xlabel('Roughness Score')
        axes[0].set_title('Score Distribution')
        labels_df['roughness_score'].plot(kind='box', ax=axes[1])
        axes[1].set_title('Box Plot')
        plt.tight_layout()
        plt.show()
        print(labels_df['roughness_score'].describe())
else:
    print("Dataset zaten mevcut, training'e geciliyor...")

In [ ]:
# Training
from train_model_v3 import train

model, history = train(batch_size=64, epochs=40)

In [ ]:
# Training Graph
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['mae'], label='Train MAE')
axes[0].plot(history.history['val_mae'], label='Val MAE')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MAE')
axes[0].set_title('Mean Absolute Error')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Huber Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Best Val MAE: {min(history.history['val_mae']):.2f}")

In [ ]:
# Validation - Her skor araligini test et
import pandas as pd
import numpy as np
import tensorflow as tf
import os

model = tf.keras.models.load_model('texture_regressor.keras')
labels_df = pd.read_csv('training_data/labels.csv')

def load_image_tf(filepath):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [224, 224])
    return tf.cast(img, tf.float32)

print("="*60)
print("SKOR ARALIGINA GORE TEST")
print("="*60)

ranges = [(0, 20), (20, 40), (40, 60), (60, 80), (80, 100)]
all_errors = []

for low, high in ranges:
    subset = labels_df[(labels_df['roughness_score'] >= low) & (labels_df['roughness_score'] < high)]
    if len(subset) < 3:
        print(f"\n{low}-{high}: Yetersiz ornek ({len(subset)})")
        continue
    
    samples = subset.sample(min(10, len(subset)), random_state=42)
    errors = []
    
    print(f"\n{low}-{high} Araligi ({len(subset)} toplam, {len(samples)} test):")
    for _, row in samples.iterrows():
        img_path = os.path.join('training_data/images', row['filename'])
        img = load_image_tf(img_path)
        pred = float(model.predict(tf.expand_dims(img, 0), verbose=0)[0][0])
        pred = np.clip(pred, 0, 100)
        error = abs(pred - row['roughness_score'])
        errors.append(error)
        all_errors.append(error)
        print(f"  {row['filename']}: Gercek={row['roughness_score']:.1f}, Tahmin={pred:.1f}, Fark={error:.1f}")
    
    print(f"  -> Aralik MAE: {np.mean(errors):.2f}")

print(f"\n{'='*60}")
print(f"GENEL MAE: {np.mean(all_errors):.2f}")
print(f"{'='*60}")

In [ ]:
# Download
from google.colab import files
import shutil

!mkdir -p export
!cp texture_regressor.keras export/
!cp training_data/pca_scaler.joblib export/
!cp training_data/labels.csv export/

shutil.make_archive('texture_model_v3', 'zip', 'export')
files.download('texture_model_v3.zip')

In [ ]:
# Test with Upload
import cv2
import numpy as np
import tensorflow as tf
from google.colab import files
import matplotlib.pyplot as plt
import os

print("Yuz fotografi yukleyin:")
uploaded = files.upload()

if uploaded:
    model = tf.keras.models.load_model('texture_regressor.keras')
    
    for filename in uploaded.keys():
        from dataset_generator_v3 import DatasetGenerator
        gen = DatasetGenerator()
        texture_map, mask = gen.process_image(filename)
        
        if texture_map is not None:
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            axes[0].imshow(cv2.imread(filename)[..., ::-1])
            axes[0].set_title('Original')
            axes[0].axis('off')
            axes[1].imshow(texture_map, cmap='gray')
            axes[1].set_title('Texture Map')
            axes[1].axis('off')
            plt.show()
            
            temp_path = '/tmp/temp_texture.png'
            cv2.imwrite(temp_path, texture_map)
            
            img = tf.io.read_file(temp_path)
            img = tf.image.decode_image(img, channels=3, expand_animations=False)
            img = tf.image.resize(img, [224, 224])
            img = tf.cast(img, tf.float32)
            
            pred = float(model.predict(tf.expand_dims(img, 0), verbose=0)[0][0])
            score = np.clip(pred, 0, 100)
            
            print(f"\n{'='*40}")
            print(f"TEXTURE SCORE: {score:.1f}/100")
            if score >= 70: print("Puruzsuz cilt")
            elif score >= 50: print("Normal doku")
            elif score >= 35: print("Orta duzey")
            else: print("Belirgin doku")
            print(f"{'='*40}")
            
            os.remove(temp_path)
        else:
            print("Yuz tespit edilemedi!")